In [0]:
%sql
DROP TABLE IF EXISTS retail_medallion_ws.default.dim_customer;
DROP TABLE IF EXISTS retail_medallion_ws.default.dim_product;

CREATE TABLE retail_medallion_ws.default.dim_customer (
    customer_sk     BIGINT,
    customer_id     STRING,
    customer_key    STRING,
    first_name      STRING,
    last_name       STRING,
    marital_status  STRING,
    gender          STRING,
    birth_date      DATE,
    country         STRING,
    row_hash        STRING,
    effective_date  DATE,
    end_date        DATE,
    is_current      BOOLEAN
)
USING DELTA;

CREATE TABLE retail_medallion_ws.default.dim_product (
    product_sk      BIGINT,
    product_id      BIGINT,
    product_key     STRING,
    product_name    STRING,
    product_cost    DOUBLE,
    product_line    STRING,
    category        STRING,
    subcategory     STRING,
    row_hash        STRING,
    effective_date  DATE,
    end_date        DATE,
    is_current      BOOLEAN
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_customer AS
SELECT
    c.cst_id AS customer_id,
    c.cst_key AS customer_key,
    c.cst_firstname AS first_name,
    c.cst_lastname AS last_name,
    CASE
        WHEN c.cst_marital_status = 'M' THEN 'Married'
        WHEN c.cst_marital_status = 'S' THEN 'Single'
        ELSE 'n/a'
    END AS marital_status,
    CASE
        WHEN UPPER(c.cst_gndr) = 'M' OR UPPER(e.GEN) = 'MALE'   THEN 'Male'
        WHEN UPPER(c.cst_gndr) = 'F' OR UPPER(e.GEN) = 'FEMALE' THEN 'Female'
        ELSE 'n/a'
    END AS gender,
    e.BDATE AS birth_date,
    COALESCE(l.CNTRY, 'n/a') AS country,
    sha2(concat_ws('||',
        coalesce(c.cst_firstname, ''),
        coalesce(c.cst_lastname, ''),
        coalesce(c.cst_marital_status, ''),
        coalesce(c.cst_gndr, ''),
        coalesce(e.GEN, ''),
        coalesce(cast(e.BDATE AS STRING), ''),
        coalesce(l.CNTRY, '')
    ), 256) AS row_hash
FROM retail_medallion_ws.default.silver_customer c
LEFT JOIN retail_medallion_ws.default.silver_erp_customer_dedup e
    ON c.cst_key = SUBSTRING(e.CID, 4, LEN(e.CID))
LEFT JOIN retail_medallion_ws.default.silver_erp_location_dedup l
    ON c.cst_key = REPLACE(l.CID, '-', '');

In [0]:
%sql
MERGE INTO retail_medallion_ws.default.dim_customer AS tgt
USING stg_customer AS src
ON tgt.customer_id = src.customer_id AND tgt.is_current = true
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN
  UPDATE SET tgt.end_date = current_date(), tgt.is_current = false;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_rows_to_insert AS
SELECT src.*
FROM stg_customer src
LEFT JOIN retail_medallion_ws.default.dim_customer tgt
    ON src.customer_id = tgt.customer_id AND tgt.is_current = true
WHERE tgt.customer_id IS NULL;

INSERT INTO retail_medallion_ws.default.dim_customer
SELECT
    (SELECT COALESCE(MAX(customer_sk), 0) FROM retail_medallion_ws.default.dim_customer)
        + ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
    customer_id, customer_key, first_name, last_name, marital_status, gender, birth_date, country,
    row_hash,
    current_date() AS effective_date,
    CAST(NULL AS DATE) AS end_date,
    true AS is_current
FROM customer_rows_to_insert;

num_affected_rows,num_inserted_rows
18484,18484


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_product AS
SELECT * FROM (
    SELECT
        p.prd_id AS product_id,
        SUBSTRING(p.prd_key, 7, LEN(p.prd_key)) AS product_key,
        p.prd_nm AS product_name,
        p.prd_cost AS product_cost,
        p.prd_line AS product_line,
        cat.CAT AS category,
        cat.SUBCAT AS subcategory,
        sha2(concat_ws('||',
            coalesce(p.prd_nm, ''),
            coalesce(cast(p.prd_cost AS STRING), ''),
            coalesce(p.prd_line, ''),
            coalesce(cat.CAT, ''),
            coalesce(cat.SUBCAT, '')
        ), 256) AS row_hash,
        ROW_NUMBER() OVER (
            PARTITION BY SUBSTRING(p.prd_key, 7, LEN(p.prd_key))
            ORDER BY p.prd_start_dt DESC
        ) AS rn
    FROM retail_medallion_ws.default.silver_product_info p
    LEFT JOIN retail_medallion_ws.default.silver_erp_category cat
        ON REPLACE(SUBSTRING(p.prd_key, 1, 5), '-', '_') = cat.ID
)
WHERE rn = 1;

In [0]:
%sql
MERGE INTO retail_medallion_ws.default.dim_product AS tgt
USING stg_product AS src
ON tgt.product_key = src.product_key AND tgt.is_current = true
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN
  UPDATE SET tgt.end_date = current_date(), tgt.is_current = false;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW product_rows_to_insert AS
SELECT src.product_id, src.product_key, src.product_name, src.product_cost,
       src.product_line, src.category, src.subcategory, src.row_hash
FROM stg_product src
LEFT JOIN retail_medallion_ws.default.dim_product tgt
    ON src.product_key = tgt.product_key AND tgt.is_current = true
WHERE tgt.product_key IS NULL;

INSERT INTO retail_medallion_ws.default.dim_product
SELECT
    (SELECT COALESCE(MAX(product_sk), 0) FROM retail_medallion_ws.default.dim_product)
        + ROW_NUMBER() OVER (ORDER BY product_key) AS product_sk,
    product_id, product_key, product_name, product_cost, product_line, category, subcategory,
    row_hash,
    current_date() AS effective_date,
    CAST(NULL AS DATE) AS end_date,
    true AS is_current
FROM product_rows_to_insert;

num_affected_rows,num_inserted_rows
295,295


In [0]:
%sql
CREATE OR REPLACE TABLE retail_medallion_ws.default.fact_sales AS
SELECT
    s.sls_ord_num AS order_id,
    p.product_sk,
    c.customer_sk,
    d.date_sk AS order_date_sk,
    s.sls_order_dt AS order_date,
    s.sls_quantity AS quantity,
    ABS(s.sls_price) AS unit_price,
    CASE
        WHEN s.sls_sales IS NULL
             OR s.sls_sales <= 0
             OR s.sls_sales <> s.sls_quantity * ABS(s.sls_price)
        THEN s.sls_quantity * ABS(s.sls_price)
        ELSE s.sls_sales
    END AS sales_amount
FROM retail_medallion_ws.default.silver_sales s
LEFT JOIN retail_medallion_ws.default.dim_product p
    ON s.sls_prd_key = p.product_key AND p.is_current = true
LEFT JOIN retail_medallion_ws.default.dim_customer c
    ON s.sls_cust_id = c.customer_id AND c.is_current = true
LEFT JOIN retail_medallion_ws.default.dim_date d
    ON try_to_date(CAST(s.sls_order_dt AS STRING), 'yyyyMMdd') = d.full_date;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT 'dim_customer' AS dim, is_current, COUNT(*) AS rows
FROM retail_medallion_ws.default.dim_customer GROUP BY is_current
UNION ALL
SELECT 'dim_product', is_current, COUNT(*)
FROM retail_medallion_ws.default.dim_product GROUP BY is_current;
SELECT customer_id, COUNT(*) AS current_versions
FROM retail_medallion_ws.default.dim_customer
WHERE is_current = true GROUP BY customer_id HAVING COUNT(*) > 1;

SELECT product_key, COUNT(*) AS current_versions
FROM retail_medallion_ws.default.dim_product
WHERE is_current = true GROUP BY product_key HAVING COUNT(*) > 1;

product_key,current_versions
